In [ ]:
from __future__ import annotations

import sys
from contextlib import contextmanager
from pathlib import Path

import pandas as pd


TARGET_COL = "Target_Log"
OUTER_FOLDS = (1, 2, 3, 4, 5)
build_pycaret_setup_kwargs = None
load_full_feature_processed = None
setup = None


In [ ]:
def resolve_current_code_dir() -> Path:
    if "__file__" in globals():
        return Path(__file__).resolve().parent

    code_dir = Path.cwd().resolve()
    if code_dir.name != "1_Code":
        raise RuntimeError(
            "cannot resolve current code directory without __file__; "
            "current working directory must be the Step5 1_Code directory"
        )
    return code_dir


def ensure_pycaret_runtime_compat() -> None:
    import numpy as np
    import scipy
    import sklearn.utils as sklearn_utils

    if not hasattr(sklearn_utils, "_print_elapsed_time"):
        @contextmanager
        def _print_elapsed_time(*args, **kwargs):
            del args, kwargs
            yield

        sklearn_utils._print_elapsed_time = _print_elapsed_time

    if not hasattr(scipy, "interp"):
        scipy.interp = np.interp


def initialize_runtime() -> Path:
    global TARGET_COL
    global build_pycaret_setup_kwargs
    global load_full_feature_processed
    global setup

    current_code_dir = resolve_current_code_dir()
    if str(current_code_dir) not in sys.path:
        sys.path.insert(0, str(current_code_dir))

    ensure_pycaret_runtime_compat()

    try:
        from pycaret.regression import setup as pycaret_setup
    except Exception as exc:  # pragma: no cover - runtime fallback for environments without pycaret
        def pycaret_setup(**kwargs):
            raise RuntimeError("pycaret.regression.setup is unavailable") from exc

    from _step5_shared import (
        TARGET_COL as shared_target_col,
        build_pycaret_setup_kwargs as shared_build_pycaret_setup_kwargs,
        load_full_feature_processed as shared_load_full_feature_processed,
    )

    TARGET_COL = shared_target_col
    build_pycaret_setup_kwargs = shared_build_pycaret_setup_kwargs
    load_full_feature_processed = shared_load_full_feature_processed
    setup = pycaret_setup
    return current_code_dir


In [ ]:
def expected_output_files() -> list[str]:
    return [
        "0_contract_check_summary.csv",
        "0_contract_check_by_fold.csv",
        "0_verified_feature_columns.csv",
    ]


def load_contract_fold_bundle(step4_root: Path, outer_fold: int) -> dict[str, object]:
    if any(dep is None for dep in (build_pycaret_setup_kwargs, load_full_feature_processed, setup)):
        initialize_runtime()

    train_df, test_df, inner_index, feature_columns = load_full_feature_processed(
        Path(step4_root),
        outer_fold=int(outer_fold),
    )
    feature_columns = list(feature_columns)
    model_train = train_df.loc[:, [*feature_columns, TARGET_COL]].copy()
    setup_kwargs = build_pycaret_setup_kwargs(
        model_train=model_train,
        target_col=TARGET_COL,
        keep_features=feature_columns,
        ignore_features=[],
        categorical_features=[],
        session_id=900 + int(outer_fold),
    )
    return {
        "outer_fold": int(outer_fold),
        "train_df": train_df,
        "test_df": test_df,
        "inner_index": inner_index,
        "feature_columns": feature_columns,
        "setup_kwargs": setup_kwargs,
    }


def validate_verified_feature_columns(
    verified_feature_columns: list[str] | None,
    current_feature_columns: list[str],
    outer_fold: int,
) -> list[str]:
    current_feature_columns = list(current_feature_columns)
    if verified_feature_columns is None:
        return current_feature_columns
    if list(verified_feature_columns) != current_feature_columns:
        raise RuntimeError(f"feature contract drift detected on outer_fold={int(outer_fold)}")
    return list(verified_feature_columns)


def collect_contract_fold_state(fold_bundle: dict[str, object]) -> dict[str, object]:
    setup(**dict(fold_bundle["setup_kwargs"]))
    train_df = pd.DataFrame(fold_bundle["train_df"])
    test_df = pd.DataFrame(fold_bundle["test_df"])
    inner_index = pd.DataFrame(fold_bundle["inner_index"])
    feature_columns = list(fold_bundle["feature_columns"])
    return {
        "outer_fold": int(fold_bundle["outer_fold"]),
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "feature_count": int(len(feature_columns)),
        "inner_index_rows": int(len(inner_index)),
        "contract_ok": True,
    }


def write_contract_outputs(
    output_dir: Path,
    contract_rows: list[dict[str, object]],
    verified_feature_columns: list[str] | None,
) -> None:
    if not contract_rows:
        raise RuntimeError("contract_rows must not be empty")
    if not verified_feature_columns:
        raise RuntimeError("verified_feature_columns must not be empty")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    by_fold_df = pd.DataFrame(contract_rows)
    summary_df = pd.DataFrame(
        [
            {
                "outer_fold_count": int(by_fold_df["outer_fold"].nunique()),
                "feature_count": int(by_fold_df["feature_count"].iloc[0]),
                "all_contract_ok": bool(by_fold_df["contract_ok"].all()),
            }
        ]
    )
    verified_features_df = pd.DataFrame({"feature": list(verified_feature_columns)})

    summary_df.to_csv(output_dir / "0_contract_check_summary.csv", index=False)
    by_fold_df.to_csv(output_dir / "0_contract_check_by_fold.csv", index=False)
    verified_features_df.to_csv(output_dir / "0_verified_feature_columns.csv", index=False)


In [ ]:
def run_contract_check(step4_root: Path, output_dir: Path) -> None:
    contract_rows: list[dict[str, object]] = []
    verified_feature_columns: list[str] | None = None

    for outer_fold in OUTER_FOLDS:
        fold_bundle = load_contract_fold_bundle(step4_root=step4_root, outer_fold=int(outer_fold))
        verified_feature_columns = validate_verified_feature_columns(
            verified_feature_columns,
            fold_bundle["feature_columns"],
            int(outer_fold),
        )
        contract_rows.append(collect_contract_fold_state(fold_bundle))

    write_contract_outputs(output_dir, contract_rows, verified_feature_columns)


In [ ]:
# --- Step 1: Resolve release-local Step4 inputs and Step5 outputs ---
current_code_dir = initialize_runtime()
step5_root = current_code_dir.parent
release_root = step5_root.parent
step4_root = release_root / "Step4_ModelInputs" / "2_Results"
output_dir = step5_root / "2_Results" / "0_input_contract_check"
contract_rows = []
verified_feature_columns = None
print({"step": "contract_runtime_ready", "step4_root": str(step4_root), "output_dir": str(output_dir)})


In [ ]:
# --- Step 2: Load and verify outer_fold=1 ---
contract_fold_1_bundle = load_contract_fold_bundle(step4_root=step4_root, outer_fold=1)
verified_feature_columns = validate_verified_feature_columns(
    verified_feature_columns,
    contract_fold_1_bundle["feature_columns"],
    1,
)
contract_fold_1_state = collect_contract_fold_state(contract_fold_1_bundle)
contract_rows.append(contract_fold_1_state)
print(contract_fold_1_state)


In [ ]:
# --- Step 3: Load and verify outer_fold=2 ---
contract_fold_2_bundle = load_contract_fold_bundle(step4_root=step4_root, outer_fold=2)
verified_feature_columns = validate_verified_feature_columns(
    verified_feature_columns,
    contract_fold_2_bundle["feature_columns"],
    2,
)
contract_fold_2_state = collect_contract_fold_state(contract_fold_2_bundle)
contract_rows.append(contract_fold_2_state)
print(contract_fold_2_state)


In [ ]:
# --- Step 4: Load and verify outer_fold=3 ---
contract_fold_3_bundle = load_contract_fold_bundle(step4_root=step4_root, outer_fold=3)
verified_feature_columns = validate_verified_feature_columns(
    verified_feature_columns,
    contract_fold_3_bundle["feature_columns"],
    3,
)
contract_fold_3_state = collect_contract_fold_state(contract_fold_3_bundle)
contract_rows.append(contract_fold_3_state)
print(contract_fold_3_state)


In [ ]:
# --- Step 5: Load and verify outer_fold=4 ---
contract_fold_4_bundle = load_contract_fold_bundle(step4_root=step4_root, outer_fold=4)
verified_feature_columns = validate_verified_feature_columns(
    verified_feature_columns,
    contract_fold_4_bundle["feature_columns"],
    4,
)
contract_fold_4_state = collect_contract_fold_state(contract_fold_4_bundle)
contract_rows.append(contract_fold_4_state)
print(contract_fold_4_state)


In [ ]:
# --- Step 6: Load and verify outer_fold=5 ---
contract_fold_5_bundle = load_contract_fold_bundle(step4_root=step4_root, outer_fold=5)
verified_feature_columns = validate_verified_feature_columns(
    verified_feature_columns,
    contract_fold_5_bundle["feature_columns"],
    5,
)
contract_fold_5_state = collect_contract_fold_state(contract_fold_5_bundle)
contract_rows.append(contract_fold_5_state)
print(contract_fold_5_state)


In [ ]:
# --- Step 7: Write contract verification outputs ---
write_contract_outputs(output_dir, contract_rows, verified_feature_columns)
print({"step": "contract_outputs_written", "files": expected_output_files(), "row_count": len(contract_rows)})
